# 🧬 Genome Firewall — Dataset Builder (run this in Google Colab)

**You do NOT need any biology knowledge.** This notebook downloads everything and
creates the files the app needs.

## How to run
1. Open this file in **Google Colab** (colab.research.google.com → File → Upload notebook).
2. (Recommended) Runtime → Change runtime type → leave as CPU. Then **Runtime → Run all**.
3. The kernel will **restart once** after the install cell — that's normal. When it does,
   just run everything **below the "After the restart" heading** again.
4. At the very end it downloads **`genome_firewall_data.zip`**. Send that zip to your teammate.

## What it produces (inside the zip)
- `labels.csv` — which antibiotic worked/failed for each genome (the training labels)
- `features.parquet` — resistance genes present in each genome (the model inputs)
- `clusters.csv` — genome groups for the honest train/test split
- `amr_gene_info.csv` — gene → drug-class map (used for honest evidence)
- `sample_genomes/` — a few genome files for the app demo

⏱️ **Heads up:** the gene-detection step runs a bio tool once per genome (~15–30s each).
Start with `QUICK_TEST = True` (20 genomes, ~10 min) to confirm it all works, then set it to
`False` and run the big batch (~500 genomes, a couple of hours).


In [ ]:
# @title 1 · CONFIG  (edit here if needed, then run)
SPECIES              = "Klebsiella pneumoniae"      # one species only
DRUGS                = ["meropenem", "ciprofloxacin", "gentamicin", "ceftazidime"]
AMRFINDER_ORGANISM   = "Klebsiella_pneumoniae"      # from `amrfinder -l`
QUICK_TEST           = True     # True = 20 genomes (fast sanity run). Set False for the real run.
MAX_GENOMES          = 500      # cap for the real run
MASH_THRESHOLD       = 0.001    # ~99.9% identity -> same cluster (grouped split)
OUT                  = "/content/gf_out"

import os
os.makedirs(OUT, exist_ok=True)
os.makedirs(OUT + "/fasta", exist_ok=True)
os.makedirs(OUT + "/amr", exist_ok=True)
os.makedirs(OUT + "/sample_genomes", exist_ok=True)
print("Config OK. QUICK_TEST =", QUICK_TEST)


In [ ]:
# @title 2 · Install conda (this triggers ONE automatic kernel restart — normal!)
!pip install -q condacolab
import condacolab
condacolab.install()
# >>> The kernel will restart here. After it restarts, continue from the next section. <<<


## ▶️ After the restart

The kernel just restarted (you'll see a "Your session crashed/restarted" note — that's expected).
Now run **every cell below**, top to bottom.


In [ ]:
# @title 3 · Install the bio tools (AMRFinderPlus + Mash) — a few minutes
!mamba install -q -y -c conda-forge -c bioconda ncbi-amrfinderplus mash
!pip -q install pandas pyarrow tqdm
!amrfinder -u   # download the AMRFinderPlus reference database (once)
print("Tools installed.")


In [ ]:
# @title 4 · Download lab results & build labels.csv
# Pulls the master AMR phenotype table from BV-BRC (lab-measured results) and filters
# it down to our species + drugs. No biology needed - it's a big spreadsheet download.
import ssl, io, pandas as pd
from ftplib import FTP_TLS

# --- read CONFIG again (survives restart) ---
SPECIES="Klebsiella pneumoniae"; DRUGS=["meropenem","ciprofloxacin","gentamicin","ceftazidime"]
OUT="/content/gf_out"

print("Connecting to BV-BRC FTP...")
ftps=FTP_TLS("ftp.bv-brc.org"); ftps.login(); ftps.prot_p()
buf=io.BytesIO()
print("Downloading AMR phenotype table (this is the big one, ~a minute)...")
ftps.retrbinary("RETR /RELEASE_NOTES/PATRIC_genomes_AMR.txt", buf.write)
ftps.quit(); buf.seek(0)

df=pd.read_csv(buf, sep="\t", dtype=str, low_memory=False)
df.columns=[c.strip().lower() for c in df.columns]
print("Total AMR rows:", len(df), "| columns:", list(df.columns)[:8], "...")

# filter to our species + drugs
df["antibiotic"]=df["antibiotic"].str.lower().str.strip()
m = df["genome_name"].str.contains(SPECIES, case=False, na=False) & df["antibiotic"].isin(DRUGS)
d = df[m].copy()

# keep lab-measured only (drop anything without a lab typing method)
if "laboratory_typing_method" in d.columns:
    d = d[d["laboratory_typing_method"].notna() & (d["laboratory_typing_method"].str.len()>0)]

# map phenotype -> binary label (1 = resistant/fail, 0 = susceptible/work); drop Intermediate
d["resistant_phenotype"]=d["resistant_phenotype"].str.strip().str.lower()
lab_map={"resistant":1,"susceptible":0}
d = d[d["resistant_phenotype"].isin(lab_map)]
d["label"]=d["resistant_phenotype"].map(lab_map)

# one label per (genome, drug): majority vote, drop ties
g = d.groupby(["genome_id","antibiotic"])["label"].mean().reset_index()
g = g[g["label"]!=0.5]                      # drop conflicting ties
g["label"]=(g["label"]>0.5).astype(int)

labels=g.rename(columns={"antibiotic":"drug"})[["genome_id","drug","label"]]
labels.to_csv(OUT+"/labels.csv", index=False)
print("\\nSaved labels.csv:", len(labels), "rows,", labels['genome_id'].nunique(), "genomes")
print(labels.groupby(["drug","label"]).size().unstack(fill_value=0))


In [ ]:
# @title 5 · Download the genome files (FASTA)
import os, pandas as pd
from ftplib import FTP_TLS
from tqdm import tqdm
OUT="/content/gf_out"
QUICK_TEST=True   # <-- set False for the full run
MAX_GENOMES=500

labels=pd.read_csv(OUT+"/labels.csv", dtype={"genome_id":str})
ids=list(labels["genome_id"].unique())
if QUICK_TEST: ids=ids[:20]
else: ids=ids[:MAX_GENOMES]
print("Downloading", len(ids), "genomes...")

ftps=FTP_TLS("ftp.bv-brc.org"); ftps.login(); ftps.prot_p()
ok=[]
for gid in tqdm(ids):
    dst=f"{OUT}/fasta/{gid}.fna"
    if os.path.exists(dst) and os.path.getsize(dst)>0: ok.append(gid); continue
    try:
        with open(dst,"wb") as fh:
            ftps.retrbinary(f"RETR /genomes/{gid}/{gid}.fna", fh.write)
        if os.path.getsize(dst)>0: ok.append(gid)
    except Exception as e:
        pass
ftps.quit()
print("Downloaded", len(ok), "of", len(ids))

# stash a few for the app demo
import shutil
for gid in ok[:4]:
    shutil.copy(f"{OUT}/fasta/{gid}.fna", f"{OUT}/sample_genomes/{gid}.fna")
open(OUT+"/downloaded_ids.txt","w").write("\\n".join(ok))


In [ ]:
# @title 6 · Detect resistance genes -> features.parquet  (the slow step)
import os, glob, subprocess, pandas as pd
from tqdm import tqdm
OUT="/content/gf_out"; AMRFINDER_ORGANISM="Klebsiella_pneumoniae"
ids=[x for x in open(OUT+"/downloaded_ids.txt").read().splitlines() if x]

rows={}; gene_info={}
for gid in tqdm(ids):
    fa=f"{OUT}/fasta/{gid}.fna"; out=f"{OUT}/amr/{gid}.tsv"
    if not os.path.exists(out):
        cmd=["amrfinder","-n",fa,"-O",AMRFINDER_ORGANISM,"--plus","-o",out,"--threads","2"]
        try: subprocess.run(cmd,check=True,capture_output=True,text=True)
        except Exception as e:
            print("amrfinder failed for",gid, str(e)[:120]); continue
    try:
        t=pd.read_csv(out,sep="\t")
    except Exception: continue
    t.columns=[c.strip() for c in t.columns]
    # column names differ across versions; find the gene-symbol + type columns
    sym=next((c for c in t.columns if c.lower() in ("element symbol","gene symbol")), t.columns[0])
    typ=next((c for c in t.columns if c.lower() in ("type","element type")), None)
    cls=next((c for c in t.columns if c.lower()=="class"), None)
    sub=next((c for c in t.columns if c.lower()=="subclass"), None)
    amr=t[t[typ].astype(str).str.upper()=="AMR"] if typ else t
    present={}
    for _,r in amr.iterrows():
        gsym=str(r[sym])
        present[gsym]=1
        gene_info.setdefault(gsym, {"class": str(r[cls]) if cls else "", "subclass": str(r[sub]) if sub else ""})
    rows[gid]=present

X=pd.DataFrame.from_dict(rows, orient="index").fillna(0).astype(int)
X.index.name="genome_id"; X=X.reset_index()
X.to_parquet(OUT+"/features.parquet", index=False)
pd.DataFrame([{"gene":k,**v} for k,v in gene_info.items()]).to_csv(OUT+"/amr_gene_info.csv", index=False)
print("Saved features.parquet:", X.shape, "(genomes x genes)")


In [ ]:
# @title 7 · Group genomes by similarity -> clusters.csv (for the honest split)
import os, glob, subprocess, pandas as pd, itertools
OUT="/content/gf_out"; MASH_THRESHOLD=0.001
fastas=sorted(glob.glob(OUT+"/fasta/*.fna"))
ids=[os.path.splitext(os.path.basename(f))[0] for f in fastas]

clusters={}
try:
    subprocess.run(["mash","sketch","-o",OUT+"/all","-s","1000",*fastas],check=True,capture_output=True,text=True)
    dist=subprocess.run(["mash","dist",OUT+"/all.msh",OUT+"/all.msh"],check=True,capture_output=True,text=True).stdout
    parent={i:i for i in ids}
    def find(x):
        while parent[x]!=x: parent[x]=parent[parent[x]]; x=parent[x]
        return x
    def union(a,b): parent[find(a)]=find(b)
    for line in dist.splitlines():
        p=line.split("\t")
        if len(p)>=3:
            a=os.path.splitext(os.path.basename(p[0]))[0]; b=os.path.splitext(os.path.basename(p[1]))[0]
            if a in parent and b in parent and float(p[2])<MASH_THRESHOLD: union(a,b)
    labelmap={}; nxt=0
    for i in ids:
        root=find(i)
        if root not in labelmap: labelmap[root]=nxt; nxt+=1
        clusters[i]=labelmap[root]
    print("Mash clustering ->", nxt, "clusters from", len(ids), "genomes")
except Exception as e:
    print("Mash failed, falling back to one-genome-per-cluster:", str(e)[:120])
    clusters={i:n for n,i in enumerate(ids)}

pd.DataFrame([{"genome_id":k,"cluster_id":v} for k,v in clusters.items()]).to_csv(OUT+"/clusters.csv", index=False)
print("Saved clusters.csv")


In [ ]:
# @title 8 · Zip everything and download
import shutil, os
OUT="/content/gf_out"
z="/content/genome_firewall_data"
if os.path.exists(z+".zip"): os.remove(z+".zip")
# only include the small processed outputs + samples (not the raw fasta/amr folders)
stage="/content/_stage"; shutil.rmtree(stage, ignore_errors=True); os.makedirs(stage)
for f in ["labels.csv","features.parquet","clusters.csv","amr_gene_info.csv","downloaded_ids.txt"]:
    p=os.path.join(OUT,f)
    if os.path.exists(p): shutil.copy(p, stage)
shutil.copytree(OUT+"/sample_genomes", stage+"/sample_genomes")
shutil.make_archive(z,"zip",stage)
print("Created", z+".zip")
try:
    from google.colab import files; files.download(z+".zip")
except Exception:
    print("Download the file from the Colab file browser (left panel): /content/genome_firewall_data.zip")
